<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/06_unet_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 06: Segmentación con U-Net + ResNet-34 — Landslide4Sense

**Modelo:** U-Net con encoder ResNet-34 (segmentation-models-pytorch)  
**Tarea:** Segmentación pixel-level — predice máscara binaria (128×128)  
**Loss:** DiceBCELoss — óptimo para objetos pequeños y clases desbalanceadas

In [ ]:
from google.colab import drive
import os, sys, h5py, subprocess
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

packages = ['segmentation-models-pytorch', 'timm', 'h5py', 'tqdm']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

drive.mount('/content/drive')

base_path = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs  = list(base_path.glob('**/TrainData/img'))

if img_dirs:
    train_img_dir  = img_dirs[0]
    train_mask_dir = train_img_dir.parent / 'mask'
    DATA_ROOT  = train_img_dir.parent.parent
    img_list   = sorted(list(train_img_dir.glob('*.h5')))
    mask_list  = sorted(list(train_mask_dir.glob('*.h5')))
    print(f'Dataset: {DATA_ROOT}')
    print(f'Muestras: {len(img_list)}')
else:
    print('ERROR: No se encontro TrainData en Drive.')
    print('Verifica que la carpeta Landslide4Sense este en MyDrive.')


## 1. U-Net + ResNet-34

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

print(f'PyTorch : {torch.__version__}')
print(f'smp     : {smp.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

def build_unet(n_channels=14, pretrained=True):
    model = smp.Unet(
        encoder_name='resnet34',
        encoder_weights='imagenet' if pretrained else None,
        in_channels=n_channels,
        classes=1,
        activation=None,   # usamos BCEWithLogitsLoss
    )
    return model

model = build_unet(n_channels=14, pretrained=True).to(device)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nParametros totales    : {total:,}')
print(f'Parametros entrenables: {trainable:,}')

# Test forward pass
x_test = torch.randn(2, 14, 128, 128).to(device)
with torch.no_grad():
    out = model(x_test)
print(f'Input  shape: {x_test.shape}')
print(f'Output shape: {out.shape}   <- (B, 1, 128, 128) mascara pixel-level')
print('Forward pass OK')


## 2. DiceBCELoss y Dataset de Segmentación

In [ ]:
from torch.utils.data import Dataset, DataLoader

# ── DiceBCELoss ──────────────────────────────────────────────────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs   = torch.sigmoid(logits)
        flat_p  = probs.view(-1)
        flat_t  = targets.view(-1)
        intersection = (flat_p * flat_t).sum()
        return 1 - (2 * intersection + self.smooth) / (flat_p.sum() + flat_t.sum() + self.smooth)

class DiceBCELoss(nn.Module):
    def __init__(self, pos_weight=None, alpha=0.5):
        super().__init__()
        self.dice  = DiceLoss()
        self.bce   = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.alpha = alpha

    def forward(self, logits, targets):
        return self.alpha * self.dice(logits, targets) + (1 - self.alpha) * self.bce(logits, targets)

# ── Dataset de segmentacion ───────────────────────────────────────────────────
class SegmentationDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.augment    = augment

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        with h5py.File(self.img_paths[idx], 'r') as f:
            img = f['img'][:].astype(np.float32)     # (128,128,14)
        with h5py.File(self.mask_paths[idx], 'r') as f:
            mask = f['mask'][:].astype(np.float32)   # (128,128)

        # Z-score por canal
        for c in range(img.shape[2]):
            ch = img[:, :, c]
            img[:, :, c] = (ch - ch.mean()) / (ch.std() + 1e-8)

        img_t  = torch.from_numpy(img.transpose(2, 0, 1))  # (14,128,128)
        mask_t = torch.from_numpy(mask).unsqueeze(0)        # (1,128,128)

        if self.augment:
            if torch.rand(1) > 0.5:
                img_t  = torch.flip(img_t,  dims=[2])
                mask_t = torch.flip(mask_t, dims=[2])
            if torch.rand(1) > 0.5:
                img_t  = torch.flip(img_t,  dims=[1])
                mask_t = torch.flip(mask_t, dims=[1])
            if torch.rand(1) > 0.5:
                img_t  = torch.rot90(img_t,  k=1, dims=[1, 2])
                mask_t = torch.rot90(mask_t, k=1, dims=[1, 2])

        return img_t, mask_t

# Verificar
test_ds = SegmentationDataset(img_list[:4], mask_list[:4])
img_t, mask_t = test_ds[0]
print(f'Imagen shape : {img_t.shape}')
print(f'Mascara shape: {mask_t.shape}')
print(f'Pixeles positivos: {mask_t.sum().item():.0f} / {mask_t.numel()} ({mask_t.mean().item():.2%})')


## 3. Entrenamiento — 5-Fold Cross Validation

In [ ]:
import torch.optim as optim
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, jaccard_score
import json

CFG = {
    'n_folds'      : 2,
    'epochs'       : 5,
    'freeze_epochs': 2,
    'batch_size'   : 8,    # segmentacion requiere mas memoria (mascara completa)
    'lr_head'      : 1e-4,
    'lr_backbone'  : 1e-5,
    'pos_weight'   : 0.703,
    'seed'         : 42,
}

torch.manual_seed(CFG['seed'])
output_dir = Path('/content/drive/MyDrive/Landslide4Sense/results/unet_resnet34')
output_dir.mkdir(parents=True, exist_ok=True)

# Etiquetas para stratified split (basadas en si el patch tiene deslizamiento)
print('Cargando etiquetas...')
all_labels = []
for mp in tqdm(mask_list):
    with h5py.File(mp, 'r') as f:
        all_labels.append(int(f['mask'][:].max() > 0))
all_labels = np.array(all_labels)
print(f'Total: {len(all_labels)} | Positivos: {all_labels.sum()} ({all_labels.mean():.2%})')

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
fold_results = []
SEP = '='*52

for fold, (train_idx, val_idx) in enumerate(skf.split(img_list, all_labels)):
    print('\n' + SEP)
    print('FOLD ' + str(fold+1) + '/' + str(CFG['n_folds']) + '  [U-Net + ResNet-34]')
    print(SEP)

    train_imgs  = [img_list[i] for i in train_idx]
    train_masks = [mask_list[i] for i in train_idx]
    val_imgs    = [img_list[i] for i in val_idx]
    val_masks   = [mask_list[i] for i in val_idx]

    train_ds = SegmentationDataset(train_imgs, train_masks, augment=True)
    val_ds   = SegmentationDataset(val_imgs,   val_masks,   augment=False)
    train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=0, pin_memory=False)
    val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=0, pin_memory=False)

    model = build_unet(n_channels=14, pretrained=True).to(device)

    # Congelar encoder (backbone)
    for p in model.encoder.parameters():
        p.requires_grad = False

    criterion = DiceBCELoss(
        pos_weight=torch.tensor([CFG['pos_weight']]).to(device)
    )
    optimizer = optim.AdamW([
        {'params': model.encoder.parameters(), 'lr': CFG['lr_backbone']},
        {'params': model.decoder.parameters(), 'lr': CFG['lr_head']},
        {'params': model.segmentation_head.parameters(), 'lr': CFG['lr_head']},
    ], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])

    history = {
        'train_loss': [], 'val_loss': [],
        'val_f1': [], 'val_auc': [],
        'val_precision': [], 'val_recall': [],
        'val_iou': [], 'val_dice': [],
    }
    best_f1, best_epoch = 0, 0
    last_probs, last_true = [], []

    for epoch in range(CFG['epochs']):
        if epoch == CFG['freeze_epochs']:
            for p in model.encoder.parameters():
                p.requires_grad = True
            print('  >> Encoder descongelado en epoca ' + str(epoch+1))

        # Train
        model.train()
        train_loss = 0
        for imgs, masks in train_dl:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_dl)

        # Validacion
        model.eval()
        val_loss, all_probs, all_preds, all_true = 0, [], [], []
        with torch.no_grad():
            for imgs, masks in val_dl:
                imgs, masks = imgs.to(device), masks.to(device)
                logits = model(imgs)
                val_loss += criterion(logits, masks).item()
                probs = torch.sigmoid(logits).cpu().numpy().flatten()
                preds = (probs > 0.5).astype(int)
                true  = masks.cpu().numpy().flatten().astype(int)
                all_probs.extend(probs.tolist())
                all_preds.extend(preds.tolist())
                all_true.extend(true.tolist())
        val_loss /= len(val_dl)

        f1   = f1_score(all_true, all_preds, zero_division=0)
        aucs = roc_auc_score(all_true, all_probs) if len(set(all_true)) > 1 else 0.0
        prec = precision_score(all_true, all_preds, zero_division=0)
        rec  = recall_score(all_true, all_preds, zero_division=0)
        iou  = jaccard_score(all_true, all_preds, zero_division=0)
        dice = 2 * iou / (1 + iou) if iou > 0 else 0.0

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(f1)
        history['val_auc'].append(aucs)
        history['val_precision'].append(prec)
        history['val_recall'].append(rec)
        history['val_iou'].append(iou)
        history['val_dice'].append(dice)
        scheduler.step()

        if f1 > best_f1:
            best_f1, best_epoch = f1, epoch + 1
            last_probs, last_true = all_probs, all_true
            torch.save(model.state_dict(), output_dir / f'fold{fold+1}_best.pt')

        print(
            f'  Ep {epoch+1:02d}/{CFG["epochs"]} | '
            f'loss {train_loss:.4f}/{val_loss:.4f} | '
            f'F1 {f1:.4f} | IoU {iou:.4f} | Dice {dice:.4f} | AUC {aucs:.4f}'
        )

    fold_results.append({
        'fold': fold+1, 'best_f1': best_f1, 'best_epoch': best_epoch,
        'history': history, 'all_probs': last_probs,
        'all_true': np.array(last_true),
    })
    print('  Mejor F1: ' + f'{best_f1:.4f}' + ' (epoca ' + str(best_epoch) + ')')

summary = {
    'model'  : 'unet_resnet34',
    'config' : CFG,
    'folds'  : [{'fold': r['fold'], 'best_f1': r['best_f1'], 'best_epoch': r['best_epoch']} for r in fold_results],
    'mean_f1': float(np.mean([r['best_f1'] for r in fold_results])),
    'std_f1' : float(np.std([r['best_f1']  for r in fold_results])),
}
with open(output_dir / 'kfold_summary.json', 'w') as fout:
    json.dump(summary, fout, indent=2)

print('\n' + SEP)
print('U-NET COMPLETADO')
print(f'F1 medio: {summary["mean_f1"]:.4f} +/- {summary["std_f1"]:.4f}')
print(SEP)


## 4. Visualización de resultados

In [ ]:
from sklearn.metrics import (
    precision_recall_curve, roc_curve, auc,
    confusion_matrix, ConfusionMatrixDisplay
)

n_folds = len(fold_results)
fig, axes = plt.subplots(n_folds, 4, figsize=(22, 5*n_folds))
if n_folds == 1:
    axes = [axes]

summary_rows = []

for i, r in enumerate(fold_results):
    h      = r['history']
    epochs = range(1, len(h['train_loss']) + 1)

    # Loss
    axes[i][0].plot(epochs, h['train_loss'], label='Train', color='steelblue')
    axes[i][0].plot(epochs, h['val_loss'],   label='Val',   color='orange')
    axes[i][0].set_title(f'Fold {r["fold"]} — Loss (DiceBCE)')
    axes[i][0].set_xlabel('Epoca')
    axes[i][0].legend()
    axes[i][0].grid(alpha=0.3)

    # Metricas pixel-level
    axes[i][1].plot(epochs, h['val_f1'],        label='F1',        color='green')
    axes[i][1].plot(epochs, h['val_iou'],       label='IoU',       color='purple')
    axes[i][1].plot(epochs, h['val_dice'],      label='Dice',      color='teal')
    axes[i][1].plot(epochs, h['val_precision'], label='Precision', color='royalblue', linestyle='--')
    axes[i][1].plot(epochs, h['val_recall'],    label='Recall',    color='red',       linestyle='--')
    axes[i][1].axvline(r['best_epoch'], color='black', linestyle=':', alpha=0.6,
                       label=f'Mejor F1={r["best_f1"]:.3f}')
    axes[i][1].set_title(f'Fold {r["fold"]} — Metricas Pixel-Level')
    axes[i][1].set_xlabel('Epoca')
    axes[i][1].legend(fontsize=7)
    axes[i][1].grid(alpha=0.3)

    # Curva PR
    prec_c, rec_c, _ = precision_recall_curve(r['all_true'], r['all_probs'])
    pr_auc = auc(rec_c, prec_c)
    axes[i][2].plot(rec_c, prec_c, color='darkorange', lw=2, label=f'AUC-PR={pr_auc:.3f}')
    axes[i][2].axhline(np.array(r['all_true']).mean(), color='navy', linestyle='--', label='Baseline')
    axes[i][2].set_xlabel('Recall')
    axes[i][2].set_ylabel('Precision')
    axes[i][2].set_title(f'Fold {r["fold"]} — Curva PR')
    axes[i][2].legend()
    axes[i][2].grid(alpha=0.3)

    # Confusion matrix
    preds = (np.array(r['all_probs']) > 0.5).astype(int)
    cm = confusion_matrix(r['all_true'], preds)
    ConfusionMatrixDisplay(cm, display_labels=['Fondo', 'Desliz.']).plot(
        cmap='YlOrRd', ax=axes[i][3], colorbar=False)
    axes[i][3].set_title(f'Fold {r["fold"]} — Confusion Matrix (pixel)')

    summary_rows.append({
        'Fold': r['fold'], 'F1': r['best_f1'],
        'IoU': max(h['val_iou']), 'Dice': max(h['val_dice']),
        'AUC': max(h['val_auc']), 'Prec': max(h['val_precision']),
        'Recall': max(h['val_recall']), 'Ep': r['best_epoch'],
    })

plt.suptitle('U-Net + ResNet-34 — Historial de Entrenamiento', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(output_dir / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nRESUMEN POR FOLD (pixel-level)')
print('-'*72)
print(f'{"Fold":>4} | {"F1":>6} | {"IoU":>6} | {"Dice":>6} | {"AUC":>6} | {"Prec":>6} | {"Recall":>6} | Ep.')
print('-'*72)
for row in summary_rows:
    print(f'{row["Fold"]:>4} | {row["F1"]:>6.4f} | {row["IoU"]:>6.4f} | {row["Dice"]:>6.4f} | '
          f'{row["AUC"]:>6.4f} | {row["Prec"]:>6.4f} | {row["Recall"]:>6.4f} | {row["Ep"]}')
print('-'*72)
means = {k: np.mean([r[k] for r in summary_rows]) for k in ['F1','IoU','Dice','AUC','Prec','Recall']}
print(f'{"MEAN":>4} | {means["F1"]:>6.4f} | {means["IoU"]:>6.4f} | {means["Dice"]:>6.4f} | '
      f'{means["AUC"]:>6.4f} | {means["Prec"]:>6.4f} | {means["Recall"]:>6.4f}')


## 5. Visualización de predicciones pixel-level

In [ ]:
# Visualizar predicciones del mejor modelo (fold 1)
best_model = build_unet(n_channels=14, pretrained=False).to(device)
best_model.load_state_dict(torch.load(output_dir / 'fold1_best.pt', map_location=device))
best_model.eval()

# Tomar 4 muestras del set de validacion
val_ds_vis = SegmentationDataset(val_imgs[:4], val_masks[:4], augment=False)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
col_titles = ['Imagen (Ch0 RGB aprox.)', 'Mascara Real', 'Prediccion U-Net']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=11)

for idx in range(4):
    img_t, mask_t = val_ds_vis[idx]
    with torch.no_grad():
        logit = best_model(img_t.unsqueeze(0).to(device))
        pred  = torch.sigmoid(logit).squeeze().cpu().numpy()

    # RGB aproximado con canales Sentinel-2 (R=Ch3, G=Ch2, B=Ch1)
    rgb = img_t.numpy()[[3, 2, 1], :, :].transpose(1, 2, 0)
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

    axes[idx][0].imshow(rgb)
    axes[idx][0].axis('off')

    axes[idx][1].imshow(mask_t.squeeze().numpy(), cmap='Reds', vmin=0, vmax=1)
    axes[idx][1].axis('off')

    axes[idx][2].imshow(pred, cmap='Reds', vmin=0, vmax=1)
    axes[idx][2].axis('off')

plt.suptitle('U-Net — Predicciones pixel-level (Fold 1 mejor modelo)', fontsize=13)
plt.tight_layout()
plt.savefig(output_dir / 'predictions_sample.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada en Drive')


## Resumen

Los pesos y resultados se guardan en Drive:  
`MyDrive/Landslide4Sense/results/unet_resnet34/`

- `fold{N}_best.pt` — pesos del mejor modelo por fold
- `kfold_summary.json` — resumen de métricas
- `training_history.png` — curvas de entrenamiento
- `predictions_sample.png` — predicciones visuales

Siguiente paso → `07_evaluation_comparison.ipynb`